# Notebook that retrieves API data related to tracks

## Imports

In [1]:
from _spo_utils import camel_to_snake, chunk_list

from functools import reduce
import pandas as pd
import requests

# Constants

In [2]:
# PATH_SPOTIFY = '../../data/2_processed/final_df.csv'
PATH_SPOTIFY = '../../data/2_processed/consolidated.csv'

## Reading dataset

In [3]:
df_treated = pd.read_csv(PATH_SPOTIFY)
unique_track_id_list = df_treated['track_id'].unique()

## ReccoBeats APIs - [Get multiple track](https://reccobeats.com/docs/apis/get-tracks) and [Get multiple audio features](https://reccobeats.com/docs/apis/get-audio-features)

In [4]:
headers = {'Accept': 'application/json'}

endpoints = {
    'audio_features': 'https://api.reccobeats.com/v1/audio-features?ids=',
    'multiple_track': 'https://api.reccobeats.com/v1/track?ids=',
}

frames = {key: [] for key in endpoints}

for chunk in chunk_list(data=unique_track_id_list, chunk_size=40, max_chunk=40):
    print(f'Len: {len(chunk)}')
    ids_str = ','.join(chunk)

    for key, base_url in endpoints.items():
        resp = requests.get(url=base_url + ids_str, headers=headers)
        if str(resp.status_code).startswith('2'):
            frames[key].append(pd.json_normalize(resp.json(), record_path='content'))

df_audio_features = pd.concat(frames['audio_features'], ignore_index=True)
df_multiple_track = pd.concat(frames['multiple_track'], ignore_index=True)

df_audio_features['track_id'] = df_audio_features['href'].str.rpartition('/')[2]
df_multiple_track['track_id'] = df_multiple_track['href'].str.rpartition('/')[2]

Len: 40
Len: 40
Len: 40
Len: 40
Len: 40
Len: 40
Len: 40
Len: 40
Len: 40
Len: 40
Len: 40
Len: 40
Len: 40
Len: 40
Len: 40
Len: 40
Len: 40
Len: 40
Len: 40
Len: 40
Len: 40
Len: 40
Len: 40
Len: 40
Len: 40
Len: 40
Len: 40
Len: 40
Len: 40
Len: 40
Len: 40
Len: 40
Len: 40
Len: 40
Len: 40
Len: 40
Len: 40
Len: 40
Len: 40
Len: 40
Len: 40
Len: 40
Len: 40
Len: 40
Len: 40
Len: 40
Len: 40
Len: 40
Len: 40
Len: 40
Len: 40
Len: 27


## Checking results

In [10]:
display(df_audio_features.head(5))
display(df_multiple_track.head(5))

,id,href,isrc,acousticness,danceability,energy,instrumentalness,key,liveness,loudness,mode,speechiness,tempo,valence,track_id
0,bbfd5c39-b2f2-4e04-a324-8dbc88e10336,https://open.spotify.com/track/7aLT0tLcS40Pena...,USUM71602108,0.03230,0.619,0.705,0.000000,1,0.0919,-5.335,1,0.0990,109.975,0.844,7aLT0tLcS40Penaplqu2cZ
1,023858f9-2052-4e49-a9f4-7ffc953e59ca,https://open.spotify.com/track/4ZxOuNHhpyOj4gv...,KRA401700174,0.06870,0.786,0.852,0.000000,8,0.0759,-2.687,1,0.0813,125.036,0.455,4ZxOuNHhpyOj4gv52MtQpT
2,2ca817f8-94b3-4c9e-9d5b-866d1764ecb5,https://open.spotify.com/track/04ZTP5KsCypmtCm...,USWB11800739,0.00281,0.630,0.694,0.000000,11,0.0719,-6.257,0,0.0253,97.005,0.216,04ZTP5KsCypmtCmQg5tH9R
3,2dbcb63d-e419-46f1-b7f4-33eaec863541,https://open.spotify.com/track/6YvqWjhGD8mB5QX...,USUG12100342,0.13000,0.627,0.792,0.000004,2,0.0845,-4.311,1,0.0310,119.054,0.415,6YvqWjhGD8mB5QXcbcUKtx
4,951b8a7c-6ce1-429b-ae68-5fac546354a8,https://open.spotify.com/track/72jbDTw1piOOj77...,USQX91603031,0.02150,0.653,0.658,0.000002,2,0.0939,-6.428,1,0.0304,99.990,0.219,72jbDTw1piOOj770jWNeaG


,id,trackTitle,artists,durationMs,isrc,ean,upc,href,availableCountries,popularity,track_id
0,bbfd5c39-b2f2-4e04-a324-8dbc88e10336,Greedy,[{'id': '320a87e6-a2b9-421f-9b4e-aa44f8c784bb'...,214906,USUM71602108,None,None,https://open.spotify.com/track/7aLT0tLcS40Pena...,"AR,AU,AT,BE,BO,BR,BG,CL,CO,CR,CY,CZ,DK,DO,DE,E...",64,7aLT0tLcS40Penaplqu2cZ
1,023858f9-2052-4e49-a9f4-7ffc953e59ca,As If It's Your Last,[{'id': '2e7ea4c5-9c78-487f-989d-04b309b5121a'...,213264,KRA401700174,None,None,https://open.spotify.com/track/4ZxOuNHhpyOj4gv...,"AR,AU,AT,BE,BO,BR,BG,CA,CL,CO,CR,CY,CZ,DK,DO,D...",73,4ZxOuNHhpyOj4gv52MtQpT
2,2ca817f8-94b3-4c9e-9d5b-866d1764ecb5,I'm a Mess,[{'id': '9eda3bba-8e60-4584-974a-478269bf1bf3'...,195519,USWB11800739,None,None,https://open.spotify.com/track/04ZTP5KsCypmtCm...,"AR,AU,AT,BE,BO,BR,BG,CA,CL,CO,CR,CY,CZ,DK,DO,D...",68,04ZTP5KsCypmtCmQg5tH9R
3,2dbcb63d-e419-46f1-b7f4-33eaec863541,Love Story (Taylor’s Version),[{'id': 'c7b330b5-a62e-420c-bf02-943ca6bb8746'...,235766,USUG12100342,None,None,https://open.spotify.com/track/6YvqWjhGD8mB5QX...,"AR,AU,AT,BE,BO,BR,BG,CA,CL,CO,CR,CY,CZ,DK,DO,D...",74,6YvqWjhGD8mB5QXcbcUKtx
4,951b8a7c-6ce1-429b-ae68-5fac546354a8,Paris,[{'id': 'e074334e-20b0-4f9e-a212-5b5632a4e935'...,221506,USQX91603031,None,None,https://open.spotify.com/track/72jbDTw1piOOj77...,"AR,AU,AT,BE,BO,BR,BG,CA,CL,CO,CR,CY,CZ,DK,DO,D...",72,72jbDTw1piOOj770jWNeaG


## Merging dataframes

In [22]:
# Columns we want to bring in from each auxiliary DataFrame
audio_cols = [
    'track_id', 
    'acousticness', 
    'danceability', 
    'energy',
    'instrumentalness', 
    'key', 
    'liveness', 
    'loudness',
    'mode', 
    'speechiness', 
    'tempo', 
    'valence'
]

track_cols = ['track_id', 'popularity', 'durationMs']

dfs_to_merge = [
    df_audio_features[audio_cols],
    df_multiple_track[track_cols],
]

df_final = reduce(
    lambda left, right: left.merge(right, on='track_id', how='left'),
    dfs_to_merge,
    df_treated.copy() # start with the treated DataFrame
)

### Renaming columns to fit snake_case pattern

In [23]:
df_final.columns = camel_to_snake(df_final.columns)
df_final.head(1)

,ts,platform,ms_played,conn_country,ip_addr,master_metadata_track_name,master_metadata_album_album_name,spotify_track_uri,episode_name,episode_show_name,...,instrumentalness,key,liveness,loudness,mode,speechiness,tempo,valence,popularity,duration_ms
0,2020-10-20T19:36:53Z,"Android OS 10 API 29 (samsung, SM-A307GT)",23600,BR,177.58.181.120,Pretty Savage,THE ALBUM,spotify:track:1XnpzbOGptRwfJhZgLbmSr,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [13]:
df_final.to_csv(PATH_SPOTIFY, index=False)